# Detecção de Defeitos em PCBs utilizando YOLOv11

>&nbsp;&nbsp;&nbsp;&nbsp;Alexandre Augusto Tescaro Oliveira  

## 1. Introdução e Motivação

> **Resumo do Estudo:** Este trabalho apresenta o desenvolvimento de um sistema de Visão Computacional para a detecção automática de defeitos em Placas de Circuito Impresso (PCBs), utilizando a arquitetura **YOLO11** (*You Only Look Once*). Para viabilizar a execução local e manter uma comparação consistente entre modelos, o conjunto de dados foi reduzido de forma controlada para aproximadamente 4 mil imagens de treino e cerca de 500 imagens para validação e teste, preservando a distribuição de classes. O pipeline também incorpora Data Augmentation no pré-processamento. Os resultados obtidos indicam bom potencial de aplicação em cenário industrial, com desempenho consistente tanto no conjunto de teste quanto em imagens fora da base principal.

### 1.1 Contextualização
No cenário da Indústria 4.0, a garantia de qualidade na fabricação de componentes eletrônicos é crítica para reduzir perdas, retrabalho e falhas em campo. As Placas de Circuito Impresso (PCBs) são a base de praticamente todos os dispositivos eletrônicos modernos. Com a miniaturização dos componentes, a inspeção visual tornou-se mais complexa e exige soluções automáticas mais robustas.

### 1.2 O Problema
Tradicionalmente, a inspeção de PCBs é realizada de forma manual por operadores humanos ou por algoritmos de visão clássica baseados em regras rígidas. Esses métodos apresentam limitações significativas no ambiente industrial:
>* **Fadiga Humana:** A inspeção visual repetitiva leva à fadiga, aumentando a chance de erro e inconsistência.
>* **Baixa Escalabilidade:** A inspeção manual é lenta e cria gargalos na linha de produção.
>* **Limitações da Visão Clássica:** Algoritmos tradicionais muitas vezes falham em lidar com variações de iluminação, rotação ou ruídos na imagem.

### 1.3 A Solução Proposta
Para reduzir esses problemas e automatizar o processo de inspeção, este trabalho adota Deep Learning com o modelo YOLO11, uma arquitetura de *single-stage detector* conhecida pelo equilíbrio entre precisão e velocidade de inferência em tempo real.

O objetivo é identificar e localizar, por meio de *Bounding Boxes*, seis tipos comuns de defeitos de fabricação:
>1.  **Missing Hole** (Furo faltante)
>2.  **Mouse Bite** (Mordida de rato/Falha na borda)
>3.  **Open Circuit** (Circuito aberto)
>4.  **Short** (Curto-circuito)
>5.  **Spur** (Esporão/Rebarba)
>6.  **Spurious Copper** (Cobre residual)

A aplicação proposta busca aumentar a eficiência do controle de qualidade industrial, reduzindo desperdícios de material e o risco de envio de placas defeituosas para a etapa seguinte do processo.

## 2. Análise Exploratória dos Dados (EDA)  
> Notebook pode ser encontrado em ./EDA_VC.ipynb
 
> Link de acesso ao Dataset utilizado: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset

In [ ]:
import importlib.util
import subprocess
import sys

FORCE_REINSTALL_TORCH = False
PREFER_CUDA_ON_NVIDIA = True
CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu121"

def _run_cmd(args):
    print("$", " ".join(args))
    subprocess.check_call(args)

def _module_exists(module_name):
    return importlib.util.find_spec(module_name) is not None

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

def _torch_stack_ready(needs_cuda):
    required_modules = ["torch", "torchvision", "torchaudio"]
    if not all(_module_exists(module_name) for module_name in required_modules):
        return False

    import torch

    if needs_cuda:
        return torch.cuda.is_available() and (torch.version.cuda is not None)
    return True

def _install_torch_stack(needs_cuda):
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "torch",
        "torchvision",
        "torchaudio",
    ]
    if needs_cuda:
        install_cmd += ["--index-url", CUDA_INDEX_URL]

    try:
        _run_cmd(install_cmd)
    except subprocess.CalledProcessError:
        print("Primeira tentativa falhou. Limpando stack PyTorch e tentando novamente...")
        _run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
        _run_cmd(install_cmd)

nvidia_gpu_detected = _has_nvidia_gpu()
needs_cuda = PREFER_CUDA_ON_NVIDIA and nvidia_gpu_detected

# Garante ferramentas básicas de build/instalação no venv recém-criado.
_run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

if FORCE_REINSTALL_TORCH or not _torch_stack_ready(needs_cuda):
    target_label = "CUDA 12.1 (cu121)" if needs_cuda else "CPU"
    print(f"Instalando stack PyTorch para {target_label}...")
    _install_torch_stack(needs_cuda)
    print("Stack PyTorch instalada/atualizada.")
else:
    print("Stack PyTorch já compatível com este ambiente.")

required_packages = {
    "pycocotools": "pycocotools",
    "pyyaml": "yaml",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "ultralytics": "ultralytics",
    "Pillow": "PIL",
    "numpy": "numpy",
    "certifi": "certifi",
}

missing = [pkg for pkg, module in required_packages.items() if not _module_exists(module)]
if missing:
    _run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", *missing])
    print("Dependências instaladas:", ", ".join(missing))
else:
    print("Dependências já instaladas.")

import torch

print(f"Torch: {torch.__version__} | CUDA build: {torch.version.cuda} | cuda_available={torch.cuda.is_available()}")
if needs_cuda and not torch.cuda.is_available():
    print("ATENÇÃO: GPU NVIDIA detectada, mas CUDA indisponível. Reinicie o kernel e execute novamente esta célula.")

In [2]:
# Diagnóstico rápido do runtime PyTorch
import subprocess
import torch

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

nvidia_gpu_detected = _has_nvidia_gpu()

print(f"GPU NVIDIA detectada: {nvidia_gpu_detected}")
print(f"Torch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"cuda_available: {torch.cuda.is_available()}")

if nvidia_gpu_detected and not torch.cuda.is_available():
    print("ATENÇÃO: há GPU NVIDIA, porém CUDA não está ativa. Reexecute a célula 4 e reinicie o kernel.")
elif (not nvidia_gpu_detected) and torch.cuda.is_available():
    print("Observação: CUDA ativa, mas nvidia-smi não foi detectado no PATH.")
else:
    print("Ambiente de execução coerente para seguir com o notebook.")

GPU NVIDIA detectada: True
Torch: 2.5.1+cu121
CUDA build: 12.1
cuda_available: True
Ambiente de execução coerente para seguir com o notebook.


In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path

import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
import numpy as np
import torchvision
from common_detection_protocol import evaluate_ultralytics_coco

# Validação rápida do operador NMS (detecta incompatibilidade torch/torchvision)
try:
    _ = torchvision.ops.nms(
        torch.tensor([[0.0, 0.0, 1.0, 1.0]]),
        torch.tensor([0.9]),
        0.5,
    )
    print("Operador torchvision::nms OK.")
except Exception as exc:
    raise RuntimeError(
        "Falha no operador torchvision."
    ) from exc

In [4]:
# Configuração de caminhos
PROJECT_ROOT = Path.cwd()
BASE_DIR = PROJECT_ROOT / "pcb-defect-subset-5000"
RUNS_ROOT = PROJECT_ROOT / "runs" / "detect"

PROJECT_RUN_DIR = RUNS_ROOT / "tcc_pcb_defect_detection" / "yolo11"
PROJECT_RUN_DIR.mkdir(parents=True, exist_ok=True)

train_images_dir = BASE_DIR / "train" / "images"
val_images_dir = BASE_DIR / "val" / "images"
test_images_dir = BASE_DIR / "test" / "images"

if not train_images_dir.exists():
    raise FileNotFoundError(f"Pasta de treino não encontrada: {train_images_dir}")

data_yaml = {
    "path": str(BASE_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {
        0: "mouse_bite",
        1: "spur",
        2: "missing_hole",
        3: "short",
        4: "open_circuit",
        5: "spurious_copper",
    },
}

yaml_path = BASE_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Dataset base: {BASE_DIR}")
print(f"Arquivo YAML: {yaml_path}")
print(f"Diretório de saída: {PROJECT_RUN_DIR}")

Dataset base: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000
Arquivo YAML: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000/data.yaml
Diretório de saída: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/yolo11


In [5]:
# ==============================================================================
# CHECAR DATA LEAKS
# ==============================================================================

def check_filename_leakage(train_dir, val_dir):
    # Pega apenas os nomes dos arquivos
    train_files = set(os.listdir(train_dir))
    val_files = set(os.listdir(val_dir))

    print(f"Total Treino: {len(train_files)}")
    print(f"Total Validação: {len(val_files)}")

    # Checa duplicatas exatas de nome
    duplicates = train_files.intersection(val_files)
    if duplicates:
        print(f"PERIGO: {len(duplicates)} arquivos têm EXATAMENTE o mesmo nome em Treino e Validação!")
        print(list(duplicates)[:5])
    else:
        print("Nomes de arquivos exatos não se repetem.")

    #  Checa vazamento por Prefixo (Assumindo que o prefixo indica a placa de origem)
    train_prefixes = set([f.split('_')[0] for f in train_files])
    val_prefixes = set([f.split('_')[0] for f in val_files])

    prefix_leak = train_prefixes.intersection(val_prefixes)

    if prefix_leak:
        print(f"ATENÇÃO: {len(prefix_leak)} placas originais (prefixos) aparecem em AMBOS os conjuntos.")
        print(f"Exemplos: {list(prefix_leak)[:5]}")
    else:
        print("Prefixos distintos. Parece que as placas foram separadas corretamente.")


if os.path.exists(train_images_dir) and os.path.exists(val_images_dir):
    check_filename_leakage(train_images_dir, val_images_dir)

Total Treino: 3973
Total Validação: 532
Nomes de arquivos exatos não se repetem.
ATENÇÃO: 3 placas originais (prefixos) aparecem em AMBOS os conjuntos.
Exemplos: ['rotation', 'l', 'light']


## 3. Arquitetura do Modelo: YOLO11

Neste estudo, o modelo de detecção de objetos **YOLO11** é utilizado como referência principal para inspeção automática de PCBs em ambiente industrial.

O YOLO se diferencia por processar a imagem em uma única passagem, dividindo-a em regiões e prevendo simultaneamente a presença e a classe dos defeitos.

### Por que YOLO para PCBs?
A escolha desta arquitetura baseia-se em três fatores relevantes para inspeção de qualidade industrial:

>* **Velocidade:** Por ser um detector de estágio único (*single-stage*), permite verificação em milissegundos, viabilizando o uso em esteiras de produção.
>* **Detecção Multiescala:** Graças ao componente **Neck**, o modelo combina características de alta e baixa resolução. Isso é importante para bases com defeitos pequenos, como *missing holes*, e defeitos maiores, como *shorts*.
>* **Anchor-Free:** O YOLO não depende de moldes fixos de caixas e adapta-se melhor a falhas irregulares como *spurs* e *mouse bites*.

### Estrutura Simplificada
O fluxo de dados dentro do modelo segue estas etapas:

>1.  **Input:** Imagem da PCB redimensionada para 640x640.
>2.  **Backbone (CSPDarknet):** Extrai as características visuais através de camadas convolucionais.
>3.  **Neck (PANet):** Combina detalhes finos com o contexto global.
>4.  **Head:** Gera as saídas finais:
>    * *Coordenadas do Bounding Box:* `[x, y, largura, altura]`
>    * *Classe:* `[probabilidade do defeito]`

<div align="center">
  <h3>Arquitetura YOLO</h3>
  <img src="https://www.researchgate.net/publication/329038564/figure/fig2/AS:694681084112900@1542636285619/YOLO-architecture-YOLO-architecture-is-inspired-by-GooLeNet-model-for-image.ppm" width="700" alt="Diagrama YOLO">
  <p><em>Figura 1: Esquema do Backbone, Neck e Head do YOLO.</em></p>
</div>

A imagem acima representa a arquitetura inicial do YOLO. Versões mais modernas não possuem mais as camadas Fully Connected no final, tratando-se de um modelo com arquitetura totalmente convolucional. A lógica geral segue as etapas abaixo:

### 1. Entrada
* A imagem entra e sofre convoluções para reduzir o seu tamanho.
* **Efeito:** A imagem é reduzida rapidamente (*Downsample*) para 1/4 do tamanho original, transformando pixels brutos em características básicas.

### 2. Backbone (Extração com C3k2)
* A imagem passa por vários blocos **C3k2**, aplicando convoluções de tamanho variável.
* A cada estágio, uma convolução de downsampling reduz o tamanho da imagem pela metade, enquanto aumenta a profundidade dos canais.
* No final, o bloco **C2PSA** aplica mecanismos de atenção para destacar as regiões de interesse.

### 3. Neck (Fusão com PANet)
* O modelo utiliza **Upsample** e **Convoluções $1 \times 1$**.
* **Objetivo:** Misturar características profundas com características rasas.

### 4. Head (Predição)
* Aplica **Convoluções $1 \times 1$** finais para gerar os vetores de saída independentes:
    1.  Um vetor para a caixa (**Regressão**).
    2.  Um vetor para a classe (**Classificação**).

In [ ]:
# Treinamento do modelo YOLO11n
if torch.cuda.is_available():
    device_id = 0
    workers = 4
    batch_size = 16
    print(f"Executando em CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device_id = "mps"
    workers = 4
    batch_size = 16
    print("Executando em Apple Silicon MPS")
else:
    device_id = "cpu"
    workers = 2
    batch_size = 8
    print("Executando em CPU")

INPUT_SIZE = 640
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 10

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_tag = "standardized_protocol"
run_name = (
    f"{run_timestamp}_yolo11n_img{INPUT_SIZE}_e{MAX_EPOCHS}_"
    f"bs{batch_size}_seed42_{run_tag}"
)

model = YOLO("yolo11n.pt")
print("Iniciando treinamento YOLO11n")
print(f"Nome do experimento: {run_name}")

results = model.train(
    data=str(yaml_path),
    epochs=MAX_EPOCHS,
    patience=EARLY_STOP_PATIENCE,
    imgsz=INPUT_SIZE,
    batch=batch_size,
    project=str(PROJECT_RUN_DIR),
    name=run_name,
    workers=workers,
    lr0=0.001,
    device=device_id,
    augment=True,
    verbose=True,

    # Otimização específica da arquitetura
    optimizer="AdamW",
    weight_decay=0.0005,

    # Augmentation moderado compartilhado com os detectores TorchVision
    hsv_h=0.01,
    hsv_s=0.20,
    hsv_v=0.20,
    degrees=10.0,
    translate=0.05,
    scale=0.10,
    fliplr=0.50,
    flipud=0.50,
    perspective=0.0,
    mosaic=0.0,
    mixup=0.0,
    erasing=0.0,

    seed=42,
    deterministic=True,
    amp=False,
    cache=False,
    pretrained=True,
    val=True,
)

results_dir = Path(results.save_dir) if hasattr(results, "save_dir") else Path(str(results))
print(f"Treinamento concluído. Resultados em: {results_dir}")

## 4. Análise de Métricas de Desempenho

Para validar a eficácia do modelo YOLO na detecção de defeitos em Placas de Circuito Impresso (PCBs), utiliza-se um conjunto robusto de métricas de avaliação. A seguir, cada métrica é interpretada no contexto de inspeção industrial.

## Matriz de Confusão
A **Matriz de Confusão** é a ferramenta fundamental para visualizar os erros do modelo. Ela compara, classe por classe, a previsão do modelo versus a realidade (Ground Truth).

* **Definição Técnica:** Uma tabela onde as linhas representam as classes reais e as colunas representam as classes preditas. A diagonal principal indica os acertos.
* **Contexto Industrial:** A matriz permite identificar "confusões funcionais" no processo de inspeção.
    * *Exemplo Crítico:* Se o modelo confundir `mouse_bite` com `open_circuit`, o erro é menos grave, pois ambos indicam falha de continuidade.
    * *Exemplo Grave:* Se o modelo classificar um defeito `short` como `background`, há um **falso negativo**, o que significa que uma placa defeituosa seguiria adiante no processo.

## Precisão (Precision)
A precisão responde à pergunta: **"De todos os defeitos que o modelo apontou, quantos eram reais?"**

$$\text{Precision} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Positivos}}$$

* **Impacto Industrial:** Se a precisão for baixa, a linha de produção poderá descartar muitas placas boas, gerando desperdício de material e custos desnecessários.

## Recall (Sensibilidade)
O Recall responde à pergunta: **"De todos os defeitos que existiam na placa, quantos o modelo conseguiu encontrar?"**

$$\text{Recall} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Negativos}}$$

* **No contexto industrial:** Esta é uma das métricas mais críticas para controle de qualidade.
* **Impacto Industrial:** Um baixo Recall significa que o modelo está deixando passar defeitos ("escapes"). Isso resulta no envio de placas defeituosas para etapas seguintes, com potencial impacto em custo, retrabalho e confiabilidade do processo.
* **Resultados recentes:** A última execução validada apresentou Recall médio de **0.9713**, com destaque para a classe crítica `missing_hole`, que permaneceu próxima de **1.00**.

## F1-Score
O F1-Score é a média harmônica entre Precisão e Recall. Ele resume a qualidade do modelo em um único número, penalizando modelos desequilibrados (ex: que acham tudo mas erram muito, ou que são precisos mas não acham nada).

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

* **Contexto Industrial:** Busca-se um F1-Score alto (> 0.90) para garantir um sistema confiável e seguro para inspeção automatizada. Na última execução validada, o F1 máximo ficou em **0.9884**.

## mAP (Mean Average Precision)
Esta é a métrica padrão-ouro para Detecção de Objetos.

* **mAP@50:** Calcula a média da precisão considerando um acerto qualquer caixa que tenha pelo menos **50% de sobreposição (IoU)** com o defeito real.
    * *Interpretação:* Indica o quão bom o modelo é em localizar e classificar o defeito corretamente.
* **mAP@50-95:** É uma métrica mais rigorosa que faz a média de vários limiares (50% a 95%).
    * *Interpretação:* Indica o quão "perfeita" e ajustada é a caixa desenhada.
* **Contexto Industrial:** Para fins de inspeção, o **mAP@50** é o indicador mais relevante. Na última execução validada, a avaliação apresentou **mAP@50 = 0.9836** e **mAP@50-95 = 0.5229**.

## Funções de Perda (Loss Functions)
Durante o treinamento, monitoram-se três tipos de "erro" que o modelo tenta minimizar:

1.  **Box Loss (Erro de Caixa):** O quão longe a caixa prevista está da caixa real. Mede o erro de coordenadas $(x, y, w, h)$.
2.  **Cls Loss (Erro de Classe):** O quão errado o modelo estava sobre o tipo de defeito.
3.  **DFL Loss (Distribution Focal Loss):** Uma métrica auxiliar usada pelo YOLO para refinar a precisão das bordas da caixa.

**Análise das Curvas:** A convergência simultânea dessas perdas, sem aumento relevante na validação, indica aprendizado saudável e compatível com uso industrial controlado, sem sinais evidentes de *overfitting* ou *underfitting*.

In [ ]:
print("Executando avaliação COCO comum no conjunto de teste")
run_dir = Path(results.save_dir)
best_checkpoint = run_dir / "weights" / "best.pt"
test_annotations_path = test_images_dir / "test_annotations.json"

if not best_checkpoint.exists():
    raise FileNotFoundError(f"Melhor checkpoint não encontrado: {best_checkpoint}")
if not test_annotations_path.exists():
    raise FileNotFoundError(f"Anotação COCO de teste não encontrada: {test_annotations_path}")

best_model = YOLO(str(best_checkpoint))
class_names = [data_yaml["names"][index] for index in sorted(data_yaml["names"])]
metrics_summary = evaluate_ultralytics_coco(
    model=best_model,
    images_dir=test_images_dir,
    annotation_path=test_annotations_path,
    class_names=class_names,
    device=device_id,
    input_size=INPUT_SIZE,
    output_json_path=run_dir / "coco_test_predictions.json",
)
metrics_summary.update(
    {
        "run_dir": str(run_dir),
        "checkpoint": str(best_checkpoint),
        "evaluation_backend": "pycocotools.COCOeval",
    }
)

with open(run_dir / "metrics_summary.json", "w", encoding="utf-8") as file:
    json.dump(metrics_summary, file, indent=2)

print("\n--- Métricas comuns de teste ---")
print(f"mAP@50: {metrics_summary['map50']:.4f}")
print(f"mAP@50-95: {metrics_summary['map50_95']:.4f}")
print(f"Precision macro: {metrics_summary['precision_macro']:.4f}")
print(f"Recall macro: {metrics_summary['recall_macro']:.4f}")
print(f"F1 macro: {metrics_summary['f1_macro']:.4f}")
display(pd.DataFrame(metrics_summary["per_class"]))

csv_path = run_dir / "results.csv"
if csv_path.exists():
    history_df = pd.read_csv(csv_path)
    history_df.columns = history_df.columns.str.strip()
    map50_column = "metrics/mAP50(B)" if "metrics/mAP50(B)" in history_df else "metrics/mAP50"
    map95_column = "metrics/mAP50-95(B)" if "metrics/mAP50-95(B)" in history_df else "metrics/mAP50-95"

    plt.figure(figsize=(9, 5))
    plt.plot(history_df["epoch"], history_df[map50_column], label="mAP@50")
    plt.plot(history_df["epoch"], history_df[map95_column], label="mAP@50-95")
    plt.xlabel("Época")
    plt.ylabel("Métrica de validação")
    plt.title("Evolução das métricas durante o treinamento")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Testes de Inferência

A etapa de inferência foi separada para o notebook `YOLOv11_inferencia.ipynb`, mantendo este arquivo focado em preparação, treinamento e análise de métricas.

Após finalizar um novo treinamento, utilize o melhor checkpoint (`best.pt`) da execução mais recente para validar o comportamento em imagens de teste e em imagens externas.

## 6. Conclusão

O notebook está configurado para retreinar o **YOLO11n** com o protocolo experimental padronizado: entrada de 640 × 640 pixels, augmentation moderado, limite máximo de 100 épocas, early stopping e seleção do melhor checkpoint pela validação.

A avaliação final utiliza o mesmo backend `pycocotools.COCOeval` adotado para as demais arquiteturas. Os resultados numéricos anteriores foram removidos desta conclusão e devem ser preenchidos somente após a execução integral do novo treinamento.

# 7. Referências  
> Documentação YOLO: https://docs.ultralytics.com/pt/  
> Documentação Pytorch: https://docs.pytorch.org/docs/stable/index.html  
> Documentação CSP-Net: https://huggingface.co/docs/timm/models/csp-darknet  
> Stanford CNN Cheatsheet: https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-convolutional-neural-networks  
> Materiais de Aula  